In [6]:
import anndata as ad
import numpy as np
import pandas as pd

# 1. Import the featureCounts table (skipping the '#' program command header line)
fc_df = pd.read_csv("featureCounts.txt", sep="\t", comment="#")

# 2. Separate gene annotations (.var) from the 6 sample count columns
# Standard featureCounts outputs have 6 annotation columns first:
# ['Geneid', 'Chr', 'Start', 'End', 'Strand', 'Length']
annot_cols = ["Geneid", "Chr", "Start", "End", "Strand", "Length"]
var_df = fc_df[annot_cols].copy()
var_df.set_index("Geneid", inplace=True)

# Extract the 6 count columns (all columns after the 6th)
counts_df = fc_df.iloc[:, 6:].copy()
counts_df.index = var_df.index

# Clean sample column names if featureCounts used full BAM file paths
# e.g., '/path/to/Control_1h_rep1.bam' -> 'Control_1h_rep1'
counts_df.columns = [
    col.split("/")[-1].replace(".bam", "") for col in counts_df.columns
]

# 3. Transpose the count matrix so samples are rows and genes are columns
counts_T = counts_df.T

# 4. Prepare the sample-information table (.obs)
# Ensure the index matches the exact sample order of counts_T
obs_df = pd.DataFrame(
    {
        "sample_id": [
            "Ctrl_1h_rep1",
            "Ctrl_1h_rep2",
            "Ctrl_1h_rep3",
            "Treat_1h_rep1",
            "Treat_1h_rep2",
            "Treat_1h_rep3",
        ],
        "treatment": [
            "Control",
            "Control",
            "Control",
            "Treated",
            "Treated",
            "Treated",
        ],
        "time_point": ["1h", "1h", "1h", "1h", "1h", "1h"],
        "replicate": ["rep1", "rep2", "rep3", "rep1", "rep2", "rep3"],
    }
).set_index("sample_id", drop=False)

# Reorder obs_df to match counts_T (or verify they already align)
obs_df = obs_df.loc[counts_T.index]

# 5. Integrity & Validation Checks
# A. Check exact sample alignment and order
assert (
    counts_T.index == obs_df.index
).all(), "Sample names or ordering mismatch between counts and metadata!"

# B. Check uniqueness of sample and gene identifiers
assert counts_T.index.is_unique, "Sample IDs are not unique!"
assert var_df.index.is_unique, "Gene IDs are not unique!"

# C. Check for missing values, negative numbers, and non-integer counts
assert not counts_T.isna().any().any(), "Count matrix contains missing values!"
assert (counts_T.values >= 0).all(), "Count matrix contains negative values!"
assert np.issubdtype(counts_T.values.dtype, np.integer) or np.all(
    counts_T.values == counts_T.values.astype(int)
), "Counts must be raw integers (not TPM/FPKM or normalized floats)!"

# Cast explicitly to integer dtype for PyDESeq2 compatibility
counts_T = counts_T.astype(np.int64)

# 6. Build the AnnData object
adata_raw = ad.AnnData(
    X=counts_T.values,
    obs=obs_df,
    var=var_df,
    uns={
        "accession": "GSE123456",
        "source": "NCBI GEO / local featureCounts alignment",
        "description": "Bulk RNA-seq raw gene counts comparing Treated vs Control at 1h (n=3 biological replicates per group).",
    },
)

# 7. Inspect dimensions and save the untouched raw object
print(f"AnnData shape (n_obs x n_vars): {adata_raw.shape}")
print(f"Samples (rows): {adata_raw.n_obs}, Genes (columns): {adata_raw.n_vars}")
adata_raw.write_h5ad("dataset_raw_counts.h5ad")

# 8. Create a separate filtered copy for downstream PyDESeq2 analysis
adata_filtered = adata_raw.copy()
# Example filter: keep genes with at least 10 total counts across samples
keep_genes = adata_filtered.X.sum(axis=0) >= 10
adata_filtered = adata_filtered[:, keep_genes].copy()
print(f"Filtered AnnData shape for PyDESeq2: {adata_filtered.shape}")

ModuleNotFoundError: No module named 'anndata'

In [ ]:
import os    
import pickle as pkl

import numpy as np 

from pydeseq2.datasets import load_example_data  
from pydeseq2.dds import DeseqDataSet 
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats 

SAVE = False  # whether to save the outputs of this notebook

if SAVE:
    # Replace this with the path to directory where you would like results to be
    # saved
    OUTPUT_PATH = "../output_files/synthetic_example"
    os.makedirs(OUTPUT_PATH, exist_ok=True)  # Create path if it doesn't exist

ModuleNotFoundError: No module named 'pydeseq2'